# LeetCode 3674 — Minimum Operations to Equalize Array bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-equalize-array/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation, Brainteaser

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một thao tác chọn **một đoạn con** `nums[l..r]` rồi gán **mọi phần tử trong đoạn** bằng **AND bit của toàn bộ mảng** (không chỉ của đoạn); tìm số thao tác **ít nhất** để mọi phần tử bằng nhau.
2. Hiểu toán tử `&` (AND bit) và tính chất `x & y <= min(x, y)`, giá trị AND của cả mảng không lớn hơn bất kỳ phần tử nào.
3. Rút ra kết luận **brainteaser**: đáp án chỉ có thể là `0` hoặc `1` — vì sao? Chứng minh ngắn và kiểm chứng bằng **BFS** trên mảng nhỏ.
4. Viết 4 cách: BFS vét cạn trạng thái (để kiểm chứng), kiểm tra `len(set(nums)) == 1`, so sánh với phần tử đầu bằng `all`, và so sánh `min == max`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: luôn trả `1`, đếm số phần tử khác AND, trả `len(set) - 1`, và chỉ so sánh `nums[0]` với AND toàn mảng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trong **một thao tác** bạn chọn một **mảng con** (đoạn liên tiếp) `nums[l..r]` rồi **thay mỗi phần tử** trong đoạn đó bằng **AND bit của tất cả phần tử của `nums`** (toàn bộ mảng tại thời điểm đó). Trả về **số thao tác tối thiểu** để **mọi phần tử của `nums` bằng nhau**.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2]` | `1` | chọn cả mảng: `1 & 2 = 0` -> `[0,0]` |
| 2 | `[5,5,5]` | `0` | đã bằng nhau |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 10^5`.

In [1]:
nums = [1, 2]
total_and = nums[0]
for x in nums[1:]:
    total_and &= x
print("AND cả mảng =", total_and)
after = [total_and] * len(nums)          # chọn cả mảng làm đoạn
print(after, "-> mọi phần tử bằng nhau:", len(set(after)) == 1)

AND cả mảng = 0
[0, 0] -> mọi phần tử bằng nhau: True


## 3. Kiến thức nền

### 3.1. AND bit (`&`)

`a & b` so sánh từng bit: bit kết quả là `1` chỉ khi **cả hai** bit đều `1`. Ví dụ `5 & 3`: `101 & 011 = 001 = 1`. Tính chất: `a & b <= a` và `a & b <= b`; AND thêm phần tử **không bao giờ làm tăng** kết quả.

In [2]:
print(5 & 3, bin(5), bin(3), bin(5 & 3))
print(1 & 2, 6 & 6, 7 & 0)
print(all((a & b) <= min(a, b) for a in range(0, 40) for b in range(0, 40)))

1 0b101 0b11 0b1
0 6 0
True


### 3.2. AND của cả mảng

AND của cả mảng `A = nums[0] & nums[1] & ... & nums[n-1]`. Tính bằng vòng lặp hoặc `functools.reduce(operator.and_, nums)`. Mỗi bit của `A` bằng `1` khi **mọi** phần tử có bit đó bằng `1`. Do đó `A <= nums[i]` với mọi `i`, và `A` **không đổi** nếu thêm phần tử bằng `A`.

In [3]:
from functools import reduce
from operator import and_

nums = [6, 14, 7]
A = reduce(and_, nums)
print(A, bin(A), [A <= x for x in nums])
print(reduce(and_, [A, 6, 14, 7]))              # thêm A vào không đổi

6 0b110 [True, True, True]
6


### 3.3. `set`, `all`, `min`/`max`

- `len(set(nums)) == 1` khi mọi phần tử bằng nhau.
- `all(x == nums[0] for x in nums)` làm điều tương tự (dừng sớm).
- `min(nums) == max(nums)` cũng tương đương.

In [4]:
for nums in ([5, 5, 5], [1, 2], [7]):
    print(nums, len(set(nums)) == 1, all(x == nums[0] for x in nums), min(nums) == max(nums))

[5, 5, 5] True True True
[1, 2] False False False
[7] True True True


### 3.4. BFS trên không gian trạng thái

Để **kiểm chứng** một giả thuyết cho mảng nhỏ, ta coi mỗi mảng là một **trạng thái**, mỗi thao tác là một **cạnh** sang trạng thái mới. BFS từ mảng ban đầu cho số bước **ít nhất** đến một trạng thái "mọi phần tử bằng nhau". Với `n <= 5` thì số trạng thái nhỏ, chạy được.

In [5]:
from collections import deque

def neighbors(state):
    n = len(state)
    A = state[0]
    for v in state[1:]:
        A &= v
    for l in range(n):
        for r in range(l, n):
            new = list(state)
            for i in range(l, r + 1):
                new[i] = A
            yield tuple(new)

print(len(set(neighbors((1, 2, 4)))))

6


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thử vài ví dụ:

```
[5,5,5] : đã bằng nhau -> 0
[1,2]   : AND = 0, chọn cả mảng -> [0,0] -> 1 thao tác
[3,5,6] : AND = 3&5&6 = 0, chọn cả mảng -> [0,0,0] -> 1 thao tác
```

Quan sát: **thao tác "chọn cả mảng"** đặt mọi phần tử bằng `A` (AND của cả mảng). Một thao tác duy nhất luôn đủ nếu mảng chưa bằng nhau. Câu hỏi còn lại: khi nào **không cần** thao tác nào? Đúng khi mọi phần tử đã bằng nhau.

## 5. Cách 1 — Brute Force / baseline

Dùng **BFS** vét cạn: từ mảng ban đầu, thử mọi đoạn `[l, r]`, sinh mảng mới; dừng khi gặp trạng thái mọi phần tử bằng nhau, trả số bước. Đây là cách "đúng theo mô tả đề" nhưng chỉ chạy nổi với mảng **rất nhỏ** (`n <= 5`, giá trị nhỏ); ta dùng nó như **oracle** để kiểm tra các cách còn lại.

Nút thắt: số trạng thái có thể tăng nhanh theo `n` và giá trị; với `n = 100` không khả thi.

In [6]:
class SolutionBFS:
    # CHỈ dùng cho mảng nhỏ (n <= 5) để kiểm chứng
    def minOperations(self, nums):
        start = tuple(nums)
        if len(set(start)) == 1:
            return 0
        seen = {start}
        q = deque([(start, 0)])
        while q:
            state, d = q.popleft()
            for nxt in neighbors(state):
                if nxt in seen:
                    continue
                if len(set(nxt)) == 1:
                    return d + 1
                seen.add(nxt)
                q.append((nxt, d + 1))
        return -1

print(SolutionBFS().minOperations([1, 2]))
print(SolutionBFS().minOperations([5, 5, 5]))
print(SolutionBFS().minOperations([3, 5, 6, 7]))

1
0
1


**Độ phức tạp:** số trạng thái tối đa `V^n` (với `V` giá trị khác nhau), mỗi trạng thái có `n(n+1)/2` láng giềng: lũy thừa theo `n`; chỉ dùng để kiểm chứng.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì mỗi phần tử mới luôn bằng `A` (AND **toàn mảng**), một thao tác trên **cả mảng** làm mọi phần tử bằng nhau ngay. Do đó **đáp án không bao giờ quá `1`**. Và đáp án là `0` **khi và chỉ khi** mọi phần tử đã bằng nhau (không có thao tác nào thì chỉ có thể bằng nhau nếu ban đầu đã bằng nhau).

**Quy tắc "nếu… thì…":**

- Nếu mọi `nums[i]` bằng nhau -> trả `0`.
- Ngược lại -> trả `1` (chọn `l = 0`, `r = n - 1`).

Ghi chú: sau thao tác ta có mảng toàn `A`; AND của mảng mới vẫn là `A` nên mảng đã "cố định" giá trị. Không cần xét tới đoạn con nhỏ hơn.

In [7]:
def answer(nums):
    return 0 if len(set(nums)) == 1 else 1

for nums in ([1, 2], [5, 5, 5], [3, 5, 6], [7]):
    print(nums, "->", answer(nums))

[1, 2] -> 1
[5, 5, 5] -> 0
[3, 5, 6] -> 1
[7] -> 0


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) BFS vét cạn (kiểm chứng); (2) `len(set(nums)) == 1`; (3) `all(x == nums[0])`; (4) `min == max`. Cách 2-4 là cùng một ý tưởng, khác cách kiểm tra "mọi phần tử bằng nhau".

### Cách 2 — Kích thước tập hợp

`set(nums)` bỏ trùng; chỉ còn một phần tử nghĩa là mọi phần tử bằng nhau.

In [8]:
class SolutionSet:
    def minOperations(self, nums):
        return 0 if len(set(nums)) == 1 else 1

print(SolutionSet().minOperations([1, 2]))
print(SolutionSet().minOperations([5, 5, 5]))

1
0


Độ phức tạp Cách 2: Time `O(n)`, Space `O(n)` (tập hợp).

### Cách 3 — `all` so với phần tử đầu

Duyệt một lượt, so sánh với `nums[0]`, **dừng sớm** khi gặp phần tử khác. Space `O(1)`.

In [9]:
class SolutionAll:
    def minOperations(self, nums):
        first = nums[0]
        for x in nums:
            if x != first:
                return 1
        return 0

print(SolutionAll().minOperations([1, 2]))
print(SolutionAll().minOperations([5, 5, 5]))

1
0


Độ phức tạp Cách 3: Time `O(n)` (tốt nhất `O(1)` khi phần tử thứ hai đã khác), Space `O(1)`.

### Cách 4 — `min == max`

Mọi phần tử bằng nhau khi và chỉ khi giá trị nhỏ nhất bằng giá trị lớn nhất.

In [10]:
class SolutionMinMax:
    def minOperations(self, nums):
        return 0 if min(nums) == max(nums) else 1

print(SolutionMinMax().minOperations([1, 2]))
print(SolutionMinMax().minOperations([5, 5, 5]))
print(SolutionMinMax().minOperations([9]))

1
0
0


Độ phức tạp Cách 4: Time `O(n)` (hai lượt `min`, `max`), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [4, 4, 4, 9, 4]`:

| bước | `i` | `nums[i]` | `first = 4` | quyết định |
|---|---|---|---|---|
| 0 | 0 | 4 | bằng | tiếp tục |
| 1 | 1 | 4 | bằng | tiếp tục |
| 2 | 2 | 4 | bằng | tiếp tục |
| 3 | 3 | 9 | khác | trả `1` (chọn cả mảng, mọi phần tử thành `AND`) |

Kiểm tra AND: `4 & 4 & 4 & 9 & 4 = 0`; chọn cả mảng ra `[0,0,0,0,0]`.

In [11]:
def trace_equalize(nums):
    first = nums[0]
    for i, x in enumerate(nums):
        same = x == first
        print(f"i={i} nums[i]={x} first={first} -> {'bằng' if same else 'KHÁC'}")
        if not same:
            A = nums[0]
            for v in nums[1:]:
                A &= v
            print("-> 1 thao tác: chọn cả mảng, mọi phần tử thành AND =", A, "->", [A] * len(nums))
            return 1
    print("-> mọi phần tử đã bằng nhau: 0 thao tác")
    return 0

trace_equalize([4, 4, 4, 9, 4])
trace_equalize([7, 7, 7])

i=0 nums[i]=4 first=4 -> bằng
i=1 nums[i]=4 first=4 -> bằng
i=2 nums[i]=4 first=4 -> bằng
i=3 nums[i]=9 first=4 -> KHÁC
-> 1 thao tác: chọn cả mảng, mọi phần tử thành AND = 0 -> [0, 0, 0, 0, 0]
i=0 nums[i]=7 first=7 -> bằng
i=1 nums[i]=7 first=7 -> bằng
i=2 nums[i]=7 first=7 -> bằng
-> mọi phần tử đã bằng nhau: 0 thao tác


0

## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int]) -> int:
        first = nums[0]
        for x in nums:
            if x != first:
                return 1
        return 0

print(Solution().minOperations([1, 2]), Solution().minOperations([5, 5, 5]))

1 0


## 10. Vì sao thuật toán đúng

- **Cận trên `1`:** nếu chọn `l = 0`, `r = n - 1` thì mọi phần tử trở thành `A = AND(nums)`. Mảng toàn `A` thỏa yêu cầu. Vậy số thao tác tối thiểu `<= 1`.
- **Cận dưới khi chưa bằng nhau:** nếu mảng ban đầu có hai phần tử khác nhau thì `0` thao tác không đủ (mảng giữ nguyên, vẫn khác nhau). Vậy cần `>= 1`, mà ta có `<= 1`, nên đáp án `= 1`.
- **Trường hợp đã bằng nhau:** không cần thao tác, đáp án `0`.
- **Vì sao AND toàn mảng quan trọng:** nếu đề chỉ AND trong **đoạn con** thì bài phức tạp hơn; ở đây giá trị gán luôn là `A` (toàn mảng), nên một thao tác trên cả mảng là đủ.
- **Cách 2-4 tương đương:** `len(set) == 1`, mọi phần tử `== nums[0]`, `min == max` đều phát biểu "mọi phần tử bằng nhau".
- **BFS (Cách 1):** thử mọi thao tác nên cho đáp án **đúng theo định nghĩa**; test mục 12 chứng minh bằng thực nghiệm rằng nó trùng với công thức `0/1`.

## 11. Phân tích Time / Space Complexity

`n = len(nums) <= 100`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS vét cạn | rất lớn (lũy thừa theo `n`), chỉ cho `n <= 5` | rất lớn |
| 2. `len(set(nums)) == 1` | `O(n)` | `O(n)` |
| 3. `all` so với phần tử đầu | `O(n)`, thường dừng sớm | `O(1)` |
| 4. `min == max` | `O(n)` | `O(1)` |

Cách 3 vừa nhanh vừa `O(1)` bộ nhớ nên là lựa chọn nộp.

## 12. Bộ test

Test 1: ví dụ đề và edge case (tuân `1 <= len <= 100`, `1 <= x <= 10^5`).

In [13]:
impls = [SolutionSet(), SolutionAll(), SolutionMinMax(), Solution()]

def oracle(nums):
    # định nghĩa trực tiếp: tồn tại hai phần tử khác nhau?
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] != nums[j]:
                return 1
    return 0

tests = [
    ([1, 2], 1),
    ([5, 5, 5], 0),
    ([1], 0),
    ([100000], 0),
    ([100000, 99999], 1),
    ([3, 5, 6], 1),                   # AND = 0
    ([7, 7, 7, 8], 1),
    ([8, 7, 7, 7], 1),                # khác ở phần tử đầu
    ([1, 1, 1, 1, 1, 1, 2], 1),       # khác ở cuối
    ([2] * 100, 0),                   # độ dài tối đa, bằng nhau
    ([2] * 99 + [3], 1),
    ([1, 3], 1),                      # AND = 1 = nums[0] nhưng chưa bằng nhau
]
for nums, exp in tests:
    assert 1 <= len(nums) <= 100 and all(1 <= x <= 10**5 for x in nums)
    assert oracle(nums) == exp
    for im in impls:
        assert im.minOperations(list(nums)) == exp, (type(im).__name__, nums)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: dùng **BFS** làm chân lý cho mảng nhỏ (`n <= 5`, giá trị `1..15`, đúng constraints) để kiểm chứng giả thuyết `0/1`; rồi test ngẫu nhiên lớn (`n` đến `100`, giá trị đến `10^5`) cho các cách còn lại.

In [14]:
import random
random.seed(3674)

for t in range(600):
    n = random.randint(1, 5)
    nums = [random.randint(1, 15) for _ in range(n)]
    truth = SolutionBFS().minOperations(nums)
    assert truth in (0, 1)
    for im in impls:
        assert im.minOperations(list(nums)) == truth, (type(im).__name__, nums, truth)
print("600 mảng nhỏ: BFS == công thức 0/1 OK")

zero_cnt = 0
for t in range(4000):
    n = random.randint(1, 100)
    if random.random() < 0.3:
        nums = [random.randint(1, 10**5)] * n
    else:
        hi = random.choice([2, 5, 100, 10**5])
        nums = [random.randint(1, hi) for _ in range(n)]
    exp = oracle(nums)
    zero_cnt += (exp == 0)
    for im in impls:
        assert im.minOperations(list(nums)) == exp
print("4000 test ngẫu nhiên lớn OK; số ca đáp án 0 =", zero_cnt)

600 mảng nhỏ: BFS == công thức 0/1 OK


4000 test ngẫu nhiên lớn OK; số ca đáp án 0 = 1249


Test 3: tính chất — đáp án chỉ là `0` hoặc `1`; sau khi áp dụng thao tác "chọn cả mảng" thì đáp án mới là `0`; hoán vị mảng không đổi đáp án; thêm phần tử bằng `nums[0]` vào mảng đã bằng nhau giữ đáp án `0`.

In [15]:
random.seed(674)
for _ in range(1500):
    n = random.randint(1, 100)
    nums = [random.randint(1, 5) for _ in range(n)]
    ans = Solution().minOperations(nums)
    assert ans in (0, 1)
    A = nums[0]
    for v in nums[1:]:
        A &= v
    after = [A] * n
    assert Solution().minOperations(after) == 0
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert Solution().minOperations(shuffled) == ans
    if ans == 0 and n < 100:
        assert Solution().minOperations(nums + [nums[0]]) == 0
print("tính chất (0/1, sau thao tác thành 0, hoán vị, thêm phần tử bằng nhau): OK")

tính chất (0/1, sau thao tác thành 0, hoán vị, thêm phần tử bằng nhau): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: luôn trả `1`

In [16]:
def wrong_always_one(nums):
    return 1                                     # SAI: quên trường hợp đã bằng nhau

print("sai :", wrong_always_one([5, 5, 5]))
print("đúng:", Solution().minOperations([5, 5, 5]))
assert wrong_always_one([5, 5, 5]) != Solution().minOperations([5, 5, 5])

sai : 1
đúng: 0


Vì sao sai: mảng `[5,5,5]` đã bằng nhau nên cần `0` thao tác. **Sửa:** kiểm tra "mọi phần tử bằng nhau" trước.

### Lỗi 2: đếm số phần tử khác AND (nghĩ mỗi phần tử một thao tác)

In [17]:
def wrong_count_diff(nums):
    A = nums[0]
    for v in nums[1:]:
        A &= v
    return sum(1 for x in nums if x != A)        # SAI: đề cho phép đổi cả đoạn trong một thao tác

print("sai :", wrong_count_diff([1, 2]))
print("đúng:", Solution().minOperations([1, 2]))
assert wrong_count_diff([1, 2]) != Solution().minOperations([1, 2])

sai : 2
đúng: 1


Vì sao sai: `[1,2]` có AND `= 0`, cả hai phần tử khác `0` nên đếm ra `2`, trong khi chọn **cả mảng** chỉ cần `1` thao tác. Thao tác áp dụng cho **một đoạn**, không phải một phần tử.

### Lỗi 3: trả `len(set(nums)) - 1`

In [18]:
def wrong_set_minus_one(nums):
    return len(set(nums)) - 1                    # SAI: nghĩ mỗi giá trị khác cần một thao tác

print("sai :", wrong_set_minus_one([1, 2, 4]))
print("đúng:", Solution().minOperations([1, 2, 4]))
assert wrong_set_minus_one([1, 2, 4]) != Solution().minOperations([1, 2, 4])

sai : 2
đúng: 1


Vì sao sai: `[1,2,4]` có `3` giá trị khác nhau nên công thức sai cho `2`; thực tế một thao tác trên cả mảng (AND `= 0`) đưa về `[0,0,0]`. **Sửa:** đáp án chỉ `0` hoặc `1`.

### Lỗi 4: chỉ so `nums[0]` với AND toàn mảng

In [19]:
def wrong_first_vs_and(nums):
    A = nums[0]
    for v in nums[1:]:
        A &= v
    return 0 if nums[0] == A else 1              # SAI: nums[0] == AND chưa chứng tỏ mọi phần tử bằng nhau

print("sai :", wrong_first_vs_and([1, 3]))
print("đúng:", Solution().minOperations([1, 3]))
assert wrong_first_vs_and([1, 3]) != Solution().minOperations([1, 3])

sai : 0
đúng: 1


Vì sao sai: `[1,3]` có AND `= 1 = nums[0]`, nhưng `3 != 1` nên mảng chưa bằng nhau (cần `1` thao tác). Điều kiện đúng là **mọi** phần tử bằng `A`, không chỉ phần tử đầu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có thao tác phức tạp (AND, OR, XOR) trên **đoạn** nhưng kết quả gán phụ thuộc **toàn mảng** -> thường có **đáp án rất nhỏ** (`0` hoặc `1`); thử thao tác "chọn cả mảng".
- Nghi ngờ brainteaser: **BFS trên mảng nhỏ** để kiểm chứng giả thuyết trước khi tin.
- AND không bao giờ làm tăng; OR không bao giờ làm giảm; XOR hai lần triệt tiêu.
- "Mọi phần tử bằng nhau": `len(set(a)) == 1`, `min(a) == max(a)`, hoặc `all(x == a[0] for x in a)`.
- Đừng vội cài thuật toán phức tạp khi `n <= 100`, `V <= 10^5`: tìm quy luật toán học trước.
- Viết thêm test với trường hợp "gần bằng nhau" (`[1,3]`, `[2,2,3]`).

## 15. Biến thể của bài

1. Phép **OR** thay AND (cũng có đáp án `0/1` vì OR cả mảng áp lên mọi phần tử).
2. Giá trị gán là AND của **chính đoạn con** (khó hơn nhiều; cần xét cấu trúc bit).
3. Đếm **số cách** chọn một đoạn làm đáp án đúng (số đoạn đạt kết quả sau 1 thao tác).
4. Mỗi thao tác chỉ được chọn đoạn có độ dài tối đa `L`.
5. Cho `q` truy vấn hỏi đáp án trên từng đoạn `[l, r]` (dùng bảng thưa `sparse table` cho min/max).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS vét cạn | thử mọi thao tác | lũy thừa theo `n` | rất lớn | kiểm chứng quy luật trên mảng nhỏ |
| 2. `len(set) == 1` | đếm số giá trị khác nhau | `O(n)` | `O(n)` | ngắn gọn |
| 3. `all` / dừng sớm | so với phần tử đầu | `O(n)` | `O(1)` | **nên nộp** |
| 4. `min == max` | khoảng giá trị bằng 0 | `O(n)` | `O(1)` | một dòng, rõ ý |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minOperations([9, 9, 8])` là gì?

**Bài 2 (tính tay):** `nums = [12, 10]`: AND cả mảng là gì? Sau một thao tác trên cả mảng, mảng ra sao?

**Bài 3 (điền TODO):** hoàn thành `and_all(nums)` trả về AND của toàn mảng.

**Bài 4 (tự viết lại):** nếu phép là **OR** thay AND, viết `min_ops_or(nums)` (đáp án vẫn `0/1`); kiểm chứng bằng BFS với `neighbors_or`.

**Bài 5 (mini challenge):** đếm số **đoạn** `[l, r]` mà sau thao tác (với AND toàn mảng) mảng thành **bằng nhau** — chỉ khi đoạn là cả mảng hoặc mảng đã bằng nhau từ đầu. Viết `count_good_segments(nums)`.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minOperations([9, 9, 8]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def and_all(nums):
    # TODO: res = nums[0]; res &= x cho các phần tử còn lại
    return -1

1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** hỏi "mảng đã bằng nhau chưa?"; nếu chưa, một thao tác trên cả mảng đủ.

**Mức 2 (vừa):** Bài 3: vòng lặp `&=`. Bài 4: OR toàn mảng `B`, thao tác trên cả mảng đưa tất cả về `B`. Bài 5: nếu mảng đã bằng nhau thì mọi đoạn đều "tốt" (`n(n+1)/2` đoạn); ngược lại chỉ đoạn `[0, n-1]` tốt (1 đoạn).

**Mức 3 (gần đáp án):** Bài 1: `[9,9,8]` có hai giá trị khác nhau -> `1`. Bài 2: `12 & 10 = 8` (`1100 & 1010 = 1000`) -> `[8, 8]`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
assert Solution().minOperations([9, 9, 8]) == 1

# Bài 2
assert 12 & 10 == 8
assert [12 & 10] * 2 == [8, 8]

# Bài 3
def and_all(nums):
    res = nums[0]
    for x in nums[1:]:
        res &= x
    return res
assert and_all([12, 10]) == 8
assert and_all([5]) == 5
assert and_all([3, 5, 6]) == 0

# Bài 4
def min_ops_or(nums):
    return 0 if len(set(nums)) == 1 else 1

def neighbors_or(state):
    n = len(state)
    B = 0
    for v in state:
        B |= v
    for l in range(n):
        for r in range(l, n):
            new = list(state)
            for i in range(l, r + 1):
                new[i] = B
            yield tuple(new)

def bfs_or(nums):
    start = tuple(nums)
    if len(set(start)) == 1:
        return 0
    seen, q = {start}, deque([(start, 0)])
    while q:
        s, d = q.popleft()
        for nx in neighbors_or(s):
            if len(set(nx)) == 1:
                return d + 1
            if nx not in seen:
                seen.add(nx)
                q.append((nx, d + 1))
for arr in ([1, 2], [3, 3], [1, 2, 4], [5]):
    assert bfs_or(arr) == min_ops_or(arr)

# Bài 5
def count_good_segments(nums):
    n = len(nums)
    if len(set(nums)) == 1:
        return n * (n + 1) // 2
    return 1
assert count_good_segments([5, 5, 5]) == 6
assert count_good_segments([1, 2]) == 1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đáp án 0/1 khi thao tác trên cả mảng luôn làm đồng nhất":**

```python
first = nums[0]
for x in nums:
    if x != first:
        return 1          # chọn cả mảng, mọi phần tử thành AND toàn mảng
return 0                  # đã bằng nhau
```

**Checklist trước khi submit:**

- [ ] Đã hiểu AND tính trên **toàn mảng**, không phải trên đoạn con?
- [ ] Trường hợp mọi phần tử bằng nhau (kể cả `n = 1`) trả `0`?
- [ ] Ngược lại trả `1` (không phải `len(set) - 1`, không phải số phần tử khác AND)?
- [ ] Kiểm tra **mọi** phần tử, không chỉ phần tử đầu hay hai phần tử đầu?
- [ ] Đã thử `[1,3]` (AND bằng `nums[0]` nhưng chưa bằng nhau), `[5,5,5]`, `[1]`?
- [ ] Chữ ký đúng `minOperations(self, nums: List[int]) -> int`?